# Notebook 54 — Knowledge Management

Demonstrates `multigen.knowledge`: entities, facts with confidence/provenance/TTL,
knowledge graph, contradiction detection, ontology, the `KnowledgeManager` facade,
and cross-agent knowledge sharing.  No external APIs needed.

In [ ]:
import sys, importlib.util
sys.path.insert(0, '../sdk')

def load(name):
    spec = importlib.util.spec_from_file_location(
        f'multigen.{name}', f'../sdk/multigen/{name}.py')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[f'multigen.{name}'] = mod
    spec.loader.exec_module(mod)
    return mod

km = load('knowledge')
print('knowledge loaded OK')

## 1. Entity + Fact — create entity, add 3 facts with different confidence levels

In [ ]:
apple = km.Entity(
    name='Apple Inc.',
    entity_type='company',
    aliases=['AAPL', 'Apple'],
    metadata={'founded': 1976, 'headquarters': 'Cupertino, CA'},
)
print('Entity:', apple.name, '| type:', apple.entity_type)
print('Aliases:', apple.aliases)

facts = [
    km.Fact('Apple Inc.', 'revenue_2023', '$383B',
            confidence=0.99, source='annual_report', provenance='10-K filing'),
    km.Fact('Apple Inc.', 'employee_count', 164_000,
            confidence=0.90, source='linkedin_estimate'),
    km.Fact('Apple Inc.', 'market_cap_estimate', '$2.8T',
            confidence=0.70, source='analyst_note'),
]

for f in facts:
    print(f'  Fact: {f.attribute} = {f.value}  confidence={f.confidence}')
    print(f'        source={f.source}  key={f.key}  id={f.fact_id}')

## 2. KnowledgeGraph — add entities + facts + relationships, query facts, neighbours

In [ ]:
kg = km.KnowledgeGraph()

# Entities
kg.add_entity(apple)
kg.add_entity(km.Entity('Tim Cook', entity_type='person'))
kg.add_entity(km.Entity('Steve Jobs', entity_type='person'))

# Facts
for f in facts:
    kg.add_fact(f)

# Relationships
kg.add_relationship(km.Relationship('Apple Inc.', 'CEO',       'Tim Cook',   confidence=1.0, source='official'))
kg.add_relationship(km.Relationship('Apple Inc.', 'founded_by','Steve Jobs',  confidence=1.0, source='wiki'))
kg.add_relationship(km.Relationship('Tim Cook',   'works_at',  'Apple Inc.', confidence=1.0, bidirectional=True))

# Query facts above 0.80 confidence
high_conf_facts = kg.facts('Apple Inc.', min_confidence=0.80)
print('Facts with confidence >= 0.80:')
for f in high_conf_facts:
    print(f'  {f.attribute}: {f.value}  (confidence={f.confidence})')

# Neighbours of Apple Inc.
print('\nNeighbours of Apple Inc.:', kg.neighbours('Apple Inc.'))
print('Neighbours of Tim Cook   :', kg.neighbours('Tim Cook'))

## 3. ContradictionDetector — conflicting facts for the same attribute

In [ ]:
detector = km.ContradictionDetector()

fact_a = km.Fact('Apple Inc.', 'ceo', 'Tim Cook',  source='official_website', confidence=1.0)
fact_b = km.Fact('Apple Inc.', 'ceo', 'Steve Jobs', source='old_press_release', confidence=0.5)
fact_c = km.Fact('Apple Inc.', 'ceo', 'Tim Cook',  source='sec_filing', confidence=0.99)

c1 = detector.add(fact_a)
print('After fact_a:', 'No contradiction' if c1 is None else c1)

c2 = detector.add(fact_b)
print('After fact_b:', c2)

c3 = detector.add(fact_c)
print('After fact_c:', 'No new contradiction' if c3 is None else c3)

all_contradictions = detector.contradictions()
print(f'\nTotal contradictions found: {len(all_contradictions)}')
for con in all_contradictions:
    print(' ', con)

## 4. Fact TTL expiry — add fact with tiny ttl_days, sleep, check is_expired

In [ ]:
import time

# ttl_days = 0.00001 day = ~0.864 seconds — well within a brief sleep
# We set expires_at manually so the test is instantaneous.

# Create fact that expires 1 ms from now
expiring_fact = km.Fact(
    entity='Apple Inc.',
    attribute='stock_price_snapshot',
    value='$189.32',
    source='market_feed',
    confidence=1.0,
    expires_at=time.time() + 0.001,   # 1 ms TTL
)

print('is_expired immediately:', expiring_fact.is_expired)
time.sleep(0.005)   # wait 5 ms
print('is_expired after sleep :', expiring_fact.is_expired)

# Permanent fact
permanent = km.Fact('Apple Inc.', 'founded', 1976, source='wiki')
print('permanent fact is_expired:', permanent.is_expired)

## 5. Ontology — hierarchy, ancestors, subtypes, is_a

In [ ]:
onto = km.Ontology()

onto.define('organisation')
onto.define('company',  parent='organisation')
onto.define('startup',  parent='company')
onto.define('unicorn',  parent='startup')
onto.define('nonprofit', parent='organisation')

print('All types     :', onto.all_types())
print('ancestors(unicorn)    :', onto.ancestors('unicorn'))
print('ancestors(startup)    :', onto.ancestors('startup'))
print('subtypes(organisation):', onto.subtypes('organisation'))
print('subtypes(company)     :', onto.subtypes('company'))
print('is_a(unicorn, company)      :', onto.is_a('unicorn', 'company'))
print('is_a(unicorn, organisation)  :', onto.is_a('unicorn', 'organisation'))
print('is_a(nonprofit, startup)     :', onto.is_a('nonprofit', 'startup'))

## 6. KnowledgeManager facade — add_entity + add_fact + add_relationship + context_for + contradictions

In [ ]:
mgr = km.KnowledgeManager()

mgr.add_entity('TechCorp', entity_type='company', sector='technology')

mgr.add_fact('TechCorp', 'revenue_2024', '$48.7B', source='annual_report', confidence=0.99)
mgr.add_fact('TechCorp', 'headcount',    12_000,   source='hr_system',     confidence=0.95)
mgr.add_fact('TechCorp', 'ceo',          'Jane Doe', source='press_release', confidence=1.0)

# Contradicting fact for ceo
mgr.add_fact('TechCorp', 'ceo', 'John Smith', source='old_wiki', confidence=0.40)

mgr.add_relationship('TechCorp', 'listed_on', 'NASDAQ', source='sec')
mgr.add_relationship('TechCorp', 'subsidiary_of', 'HoldingCo', source='sec')

print(mgr.context_for('TechCorp', max_facts=5, min_confidence=0.9))

print('\nContradictions:')
for c in mgr.contradictions():
    print(' ', c)

## 7. Cross-agent knowledge sharing — share_fact from manager A to manager B

In [ ]:
agent_a = km.KnowledgeManager()
agent_b = km.KnowledgeManager()

agent_a.add_entity('Acme Corp', entity_type='company')
shared_fact = agent_a.add_fact(
    'Acme Corp', 'q4_revenue', '$5.2B',
    source='agent_a_research', confidence=0.88,
)

print('Agent A facts about Acme Corp:', len(agent_a.graph.facts('Acme Corp')))
print('Agent B facts about Acme Corp before share:', len(agent_b.graph.facts('Acme Corp')))

# Share the fact
agent_a.share_fact(shared_fact, agent_b)

print('Agent B facts about Acme Corp after share:', len(agent_b.graph.facts('Acme Corp')))

# Verify in B
b_facts = agent_b.graph.facts('Acme Corp')
for f in b_facts:
    print(f'  Agent B sees: {f.attribute} = {f.value}  source={f.source}  confidence={f.confidence}')